In [8]:
!pip install -q pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("Retail_ETL_MiniProject")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
print("Active Spark Version:", spark.version)

Active Spark Version: 4.0.5


In [9]:
import os, csv
os.makedirs("/content/raw_data", exist_ok=True)

orders_rows = [
    ["order_id", "customer_id", "product_id", "quantity", "unit_price", "order_ts", "status"],
    ["ORD-1001", "CUST-01", "PROD-A", "2", "49.99", "2026-10-01 08:15:00", "COMPLETED"],
    ["ORD-1002", "CUST-02", "PROD-B", "1", "120.00", "2026-10-01 09:22:10", "completed"],
    ["ORD-1002", "CUST-02", "PROD-B", "1", "120.00", "2026-10-01 09:22:10", "completed"], # Duplicate
    ["ORD-1003", "CUST-03", "PROD-C", "-1", "15.50", "2026-10-01 10:05:45", "COMPLETED"], # Invalid qty
    ["ORD-1004", "", "PROD-A", "3", "49.99", "2026-10-02 11:30:00", "COMPLETED"], # Missing customer
    ["ORD-1005", "CUST-01", "PROD-D", "1", "299.00", "2026-10-02 14:12:00", "CANCELLED"],
    ["ORD-1006", "CUST-04", "PROD-B", "2", "120.00", "2026-10-02 16:45:30", "COMPLETED"],
    ["ORD-1007", "CUST-02", "PROD-A", "4", "49.99", "2026-10-03 09:10:00", "COMPLETED"],
    ["ORD-1008", "CUST-05", "PROD-C", "5", "15.50", "2026-10-03 12:00:00", "COMPLETED"],
    ["ORD-1009", "CUST-01", "PROD-B", "1", "120.00", "2026-10-03 18:20:00", "COMPLETED"],
    ["ORD-1010", "CUST-03", "PROD-D", "2", "299.00", "invalid_date", "COMPLETED"] # Corrupt timestamp
]

products_rows = [
    ["product_id", "product_name", "category"],
    ["PROD-A", "Wireless Ergonomic Mouse", "Accessories"],
    ["PROD-B", "Mechanical Keyboard", "Peripherals"],
    ["PROD-C", "USB-C Braided Cable", "Accessories"],
    ["PROD-D", "27-Inch 4K Monitor", "Displays"]
]

with open("/content/raw_data/orders_raw.csv", "w", newline="") as f:
    csv.writer(f).writerows(orders_rows)

with open("/content/raw_data/products_raw.csv", "w", newline="") as f:
    csv.writer(f).writerows(products_rows)

print("Raw datasets written to /content/raw_data/")

Raw datasets written to /content/raw_data/


In [10]:
# Define explicit schemas
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("order_ts", StringType(), True), # Kept as string to safely catch malformed dates
    StructField("status", StringType(), True)
])

products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True)
])

# Read raw CSV files
orders_bronze_df = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv("/content/raw_data/orders_raw.csv")
)

products_bronze_df = (
    spark.read
    .option("header", True)
    .schema(products_schema)
    .csv("/content/raw_data/products_raw.csv")
)

# Add audit tracking columns
orders_bronze_df = (
    orders_bronze_df
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

products_bronze_df = (
    products_bronze_df
    .withColumn("ingested_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

# Save to Bronze Parquet storage
orders_bronze_df.write.mode("overwrite").parquet("/content/lakehouse/bronze/orders")
products_bronze_df.write.mode("overwrite").parquet("/content/lakehouse/bronze/products")

print("Bronze layer ingested successfully!")

Bronze layer ingested successfully!


In [16]:
# Read back Bronze tables
orders_df = spark.read.parquet("/content/lakehouse/bronze/orders")
products_df = spark.read.parquet("/content/lakehouse/bronze/products")

# Cleanse and standardize orders data using try_to_timestamp to handle invalid dates safely
cleaned_orders = (
    orders_df
    .dropDuplicates(["order_id"]) # Remove duplicates
    .filter(F.col("customer_id").isNotNull() & (F.col("customer_id") != "")) # Remove missing customer IDs
    .filter(F.col("quantity") > 0) # Remove invalid quantities
    .filter(F.col("unit_price") > 0) # Remove invalid prices
    .withColumn("status", F.upper(F.trim(F.col("status")))) # Standardize status casing
    .withColumn("parsed_ts", F.try_to_timestamp(F.col("order_ts"), F.lit("yyyy-MM-dd HH:mm:ss"))) # Safely parse timestamp
    .filter(F.col("parsed_ts").isNotNull()) # Drop unparseable/null timestamps
    .drop("order_ts") # Drop the old string column to avoid ambiguity
    .withColumnRenamed("parsed_ts", "order_ts") # Rename parsed_ts to order_ts
)

# Add derived columns
#Calculates total line revenue (quantity × unit_price) rounded to 2 decimals, and extracts the date part (order_date) for partitioning later.
cleaned_orders = (
    cleaned_orders
    .withColumn("line_total", F.round(F.col("quantity") * F.col("unit_price"), 2))
    .withColumn("order_date", F.to_date(F.col("order_ts")))
)

# Broadcast join with product lookup table
#Broadcast Join: Performs a left join with products. Wrapping the small products table in F.
#broadcast() sends a copy of the product catalog to all worker nodes, eliminating heavy data shuffling across the network and speeding up execution.
enriched_orders = (
    cleaned_orders
    .join(
        F.broadcast(products_df.select("product_id", "product_name", "category")),
        on="product_id",
        how="left"
    )
)

# Save to Silver layer
enriched_orders.write.mode("overwrite").parquet("/content/lakehouse/silver/enriched_orders")

print("Silver layer processed successfully! Clean records count:")
enriched_orders.show(10, truncate=False)

Silver layer processed successfully! Clean records count:
+----------+--------+-----------+--------+----------+---------+--------------------------+---------------------------------------+-------------------+----------+----------+------------------------+-----------+
|product_id|order_id|customer_id|quantity|unit_price|status   |ingested_at               |source_file                            |order_ts           |line_total|order_date|product_name            |category   |
+----------+--------+-----------+--------+----------+---------+--------------------------+---------------------------------------+-------------------+----------+----------+------------------------+-----------+
|PROD-A    |ORD-1001|CUST-01    |2       |49.99     |COMPLETED|2026-11-17 14:45:01.835823|file:///content/raw_data/orders_raw.csv|2026-10-01 08:15:00|99.98     |2026-10-01|Wireless Ergonomic Mouse|Accessories|
|PROD-B    |ORD-1002|CUST-02    |1       |120.0     |COMPLETED|2026-11-17 14:45:01.835823|file:///cont

In [17]:
# Read Silver enriched table
silver_df = spark.read.parquet("/content/lakehouse/silver/enriched_orders")

# Exclude CANCELLED orders for business reporting
active_orders = silver_df.filter(F.col("status") != "CANCELLED")

# --- Gold Table 1: Daily Category Performance ---
daily_category_sales = (
    active_orders
    .groupBy("order_date", "category")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_units_sold"),
        F.sum("line_total").alias("gross_revenue")
    )
)

# Save partitioned by order_date
#Partitioning: Writes the table to disk partitioned by order_date, creating organized Hive-style folders
daily_category_sales.write.mode("overwrite").partitionBy("order_date").parquet("/content/lakehouse/gold/daily_category_sales")

# --- Gold Table 2: Customer Spend Leaderboard ---
customer_agg = (
    active_orders
    .groupBy("customer_id")
    .agg(
        F.sum("line_total").alias("total_spend"),
        F.countDistinct("order_id").alias("order_count")
    )
)

# Window function to rank customers by total spend
window_spec = Window.orderBy(F.desc("total_spend"))

customer_rankings = (
    customer_agg
    .withColumn("spend_rank", F.dense_rank().over(window_spec))
)

customer_rankings.write.mode("overwrite").parquet("/content/lakehouse/gold/customer_rankings")

print("Gold tables generated successfully!")

Gold tables generated successfully!


In [18]:
print("--- 1. Execution Plan (Check for BroadcastHashJoin) ---")
enriched_orders.explain("formatted")

print("\n--- 2. Partition Directory Structure Check ---")
!ls -R /content/lakehouse/gold/daily_category_sales

print("\n--- 3. Viewing Gold Table 1: Daily Category Sales ---")
spark.read.parquet("/content/lakehouse/gold/daily_category_sales").show(truncate=False)

print("\n--- 4. Viewing Gold Table 2: Customer Rankings ---")
spark.read.parquet("/content/lakehouse/gold/customer_rankings").show(truncate=False)

--- 1. Execution Plan (Check for BroadcastHashJoin) ---
== Physical Plan ==
AdaptiveSparkPlan (15)
+- Project (14)
   +- BroadcastHashJoin LeftOuter BuildRight (13)
      :- Project (9)
      :  +- Project (8)
      :     +- Filter (7)
      :        +- SortAggregate (6)
      :           +- Sort (5)
      :              +- Exchange (4)
      :                 +- SortAggregate (3)
      :                    +- Sort (2)
      :                       +- Scan parquet  (1)
      +- BroadcastExchange (12)
         +- Filter (11)
            +- Scan parquet  (10)


(1) Scan parquet 
Output [9]: [order_id#253, customer_id#254, product_id#255, quantity#256, unit_price#257, order_ts#258, status#259, ingested_at#260, source_file#261]
Batched: true
Location: InMemoryFileIndex [file:/content/lakehouse/bronze/orders]
ReadSchema: struct<order_id:string,customer_id:string,product_id:string,quantity:int,unit_price:double,order_ts:string,status:string,ingested_at:timestamp,source_file:string>

(2) Sort